In [ ]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] | {tx['user_id']} | {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(0.5)

producer.flush()
producer.close()

In [ ]:
%%file consumer.py
from kafka import KafkaConsumer
import json
from collections import defaultdict
from datetime import datetime, timedelta

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda m: json.loads(m.decode('utf-8')),
    auto_offset_reset='latest'
)

user_history = defaultdict(list)

print("Detektor anomalii uruchomiony. Monitorowanie transakcji...")

try:
    for message in consumer:
        tx = message.value
        user_id = tx['user_id']

        current_tx_time = datetime.fromisoformat(tx['timestamp'])
        
        user_history[user_id].append(current_tx_time)
        
        threshold = current_tx_time - timedelta(seconds=60)
        user_history[user_id] = [t for t in user_history[user_id] if t > threshold]
        
        tx_count = len(user_history[user_id])
        if tx_count > 3:
            print(f"⚠️  ALERT: Wykryto serię transakcji!")
            print(f"   Użytkownik: {user_id}")
            print(f"   Liczba operacji: {tx_count} w ciągu ostatnich 60s")
            print(f"   Ostatnia transakcja: {tx['tx_id']} ({tx['amount']} PLN)\n")
        else:
            print(f"OK: {tx['tx_id']} dla {user_id} (Suma w oknie: {tx_count})")

except KeyboardInterrupt:
    print("\nZatrzymywanie konsumenta...")
finally:
    consumer.close()